In [3]:
import moviepy as mp
import cv2
import pytesseract
from sklearn.cluster import KMeans
import numpy as np
import matplotlib.pyplot as plt
import warnings

# Ignore warnings about all-NaN slices of NumPy arrays
warnings.filterwarnings('ignore', message='All-NaN slice encountered')


def get_mean_background(video, video_fname):
    """
    Get the mean background value of pixels in video by averaging over frames.
    """
    for tt in (rng:=range(0, int(video.duration), 25)):
        img = get_monochrome(video.to_ImageClip(t=tt).get_frame(0))
        img = isolate_window(img, video_fname=='dash_helpers' and tt < 50)
        if tt == 0:
            meanval = blur_img(img, 25)
        else:
            meanval += blur_img(img, 25)
    return blur_img(meanval/len(rng), 125)


def blur_img(img, ksize):
    """
    Apply a Gaussian blur to the image.
    """
    return cv2.GaussianBlur((img * 255).astype(np.uint8), (ksize,ksize), 0) / 255


def get_monochrome(img):
    """
    Convert an RGB image to monochrome by averaging the color channels and normalizing to [0, 1]
    """
    return np.mean(img.astype(float), axis=2) / 255


def isolate_window(img, firstfew=False, w=1680, h=1050):
    """
    Isolate the window region in the image by applying a perspective transform.
    """
    corners = ([[300,255], [291,1684], [1080,1667], [1080,284]] if firstfew
               else [[165,177], [152,1740], [1034,1684], [1034,245]])
    pts_src = np.flip(np.array(corners, dtype=np.float32), 1)
    pts_dst = np.array([[0, 0], [w, 0], [w, h], [0, h]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(pts_src, pts_dst)
    return cv2.warpPerspective(img, M, (w, h))


def subtract_background(img, background):
    """
    Subtract mean background from image.
    """
    return (1+img-background)


def binarize_with_kmeans(img, blocksize):
    """
    Binarize an image using K-means clustering on blocks.
    """
    img = np.copy(img)
    kmeans = KMeans(n_clusters=2, random_state=0)
    for y in range(0,img.shape[0],blocksize):
        for x in range(0,img.shape[1],blocksize):
            if np.ptp(s:=img[y:y+blocksize, x:x+blocksize]) > 0.1:
                kmeans.fit(s.reshape(-1, 1))
                cutoff = np.mean(kmeans.cluster_centers_)
                img[y:y+blocksize, x:x+blocksize] = (s > cutoff).astype(float)
    return img


def get_text(img):
    """
    Extract text from an image using pytesseract.
    """
    return pytesseract.image_to_string(np.round(img*255).astype(np.uint8))


def raw_txt_from_video(fname, path, write_to_file=True):
    """
    Extract and compile raw text from distinct frames of video file.
    """
    video = mp.VideoFileClip(f'{path}/{fname}.mp4')
    mean_bg = get_mean_background(video, fname)
    text = []
    for tt in range(int(video.duration)):
        print(f"\rProcessing frame {tt+1} of {int(video.duration)} ", end='')
        img = get_monochrome(video.to_ImageClip(t=tt).get_frame(0))
        img = isolate_window(img, fname=='dash_helpers' and tt < 50)
        imgb = blur_img(img, 5)
        if tt == 0 or np.mean(np.abs(imgb-imgb_prev) > 0.1) > 0.01:
            img = subtract_background(img, mean_bg)
            img = binarize_with_kmeans(img, 100)
            text.append(get_text(img))
        imgb_prev = imgb
    text = '\n'.join(text)
    if write_to_file:
        with open(f"output/rawtxt_{fname}.txt", "w") as f:
            f.write(text)
    return text

In [9]:
path_to_videos = '/mnt/c/Users/gavin/Documents/Effo Code Videos'
raw_txt_from_video('corebcc_faq_app', path_to_videos)

Processing frame 486 of 486 

'- OSPek (C) > Users » 9977209 > git > carebec-taq-app > oo ~ U Search corebcc-faq-app\n\nName 3 Date modified\n\na\n‘ j OU 64s) [era mane 6713/2024 3.33 PM\nBogut : ; ,)U2e/2034 5.38 AM\nB github es. 9 Fie tokdes\nB idea eos we File folder i.\nB app m3 File roider Fim\nBi nei Fie fokler Bin ee\nB venv ; : Fée fobder a\n& guignore Barer ort ae Tent Document\nLo aocker-compene yma my,\nL’ Dockerfile\n@ README ma\n\n\n@ OSOek (C) > Users > 9977209 > git > corebcc-faq-app ~ U Search corebce -fag-app\n-\nal\n\n- or\n\nHow do you want to open this file?\n\nInternet Explorer\n\nOffice XML Handler\n\nPaint\n\nUs| PyCharm 202213 age See\n\n6 | Windows Media Player\n| Word a ae\nry WordPad\n\n\n© QSDwk (C) * Users > 9977209 > git > corebcc-faq-app rome. v Us Search corebcc-faq-app\n\na oa, Lt nell ce -\nvr) ir ae —*\n™ How do you want to open this file?\n\na | Internet Explorer\n\np BUILD STAGE\nFROM docker-all.artifactory.8451.cloud/core-images/pytnon:3.9-build as buila\n\nARG PYTHON_PACKAGE_R